# SleepSense — TFLite Export & Validation
Converts the trained Keras SavedModel → TFLite (INT8 quantized, <5 MB)  
then validates accuracy is within 2% of the full-precision model.  
The output `snore_classifier.tflite` drops directly into `mobile/assets/models/`.

In [ ]:
import os, json
import numpy as np
import tensorflow as tf
from sklearn.metrics import f1_score

MODEL_DIR  = '/content/models'
TFLITE_OUT = os.path.join(MODEL_DIR, 'snore_classifier.tflite')
SPEC_DIR   = '/content/data/spectrograms'
IMG_SIZE   = 128
CLASSES    = ['snoring', 'breathing', 'silence', 'ambient']

print(f'TF {tf.__version__}')

## 1. Load trained model

In [ ]:
model = tf.keras.models.load_model(os.path.join(MODEL_DIR, 'snore_classifier_savedmodel'))
print('Model loaded.')
model.summary(line_length=80)

## 2. Representative dataset for INT8 calibration

In [ ]:
# Use 200 random spectrograms from training set to calibrate quantization
all_specs = sorted([f for f in os.listdir(SPEC_DIR) if f.endswith('.npy')])
calib_files = np.random.choice(all_specs, size=min(200, len(all_specs)), replace=False)

def representative_dataset():
    for fname in calib_files:
        spec = np.load(os.path.join(SPEC_DIR, fname))
        # Replicate to 3 channels, add batch dim
        spec3 = np.stack([spec, spec, spec], axis=-1)[np.newaxis].astype(np.float32)
        yield [spec3]

print(f'Calibration set: {len(calib_files)} samples')

## 3. Convert — INT8 post-training quantization

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32   # keep float I/O for react-native-fast-tflite
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

with open(TFLITE_OUT, 'wb') as f:
    f.write(tflite_model)

size_mb = os.path.getsize(TFLITE_OUT) / 1024 / 1024
print(f'TFLite model saved: {TFLITE_OUT}')
print(f'Size: {size_mb:.2f} MB  (target <5 MB)')

if size_mb > 5:
    print('⚠️  Model exceeds 5 MB — consider further pruning or a smaller backbone.')
else:
    print('✅  Size OK')

## 4. Validate — accuracy must stay within 2% of full-precision

In [ ]:
# Load test split saved from notebook 01
test_indices_path = os.path.join(MODEL_DIR, 'test_indices.npy')
# If this file doesn't exist, re-run the split cell from notebook 01 and save indices:
#   np.save(os.path.join(MODEL_DIR, 'test_indices.npy'), np.array(list(zip(X_test, y_test)), dtype=object))

# For a quick validation, sample 100 test files from SPEC_DIR
sample_files = np.random.choice(all_specs, size=100, replace=False)

# ── Full-precision predictions ────────────────────────────────────────────────
fp_preds  = []
fp_labels = []
for fname in sample_files:
    spec  = np.load(os.path.join(SPEC_DIR, fname))
    spec3 = np.stack([spec, spec, spec], axis=-1)[np.newaxis].astype(np.float32)
    pred  = np.argmax(model.predict(spec3, verbose=0))
    # label encoded in filename prefix as first digit (set during notebook 01)
    fp_preds.append(pred)

# ── TFLite predictions ────────────────────────────────────────────────────────
interp = tf.lite.Interpreter(model_path=TFLITE_OUT)
interp.allocate_tensors()
inp_det = interp.get_input_details()[0]
out_det = interp.get_output_details()[0]

tflite_preds = []
for fname in sample_files:
    spec  = np.load(os.path.join(SPEC_DIR, fname))
    spec3 = np.stack([spec, spec, spec], axis=-1)[np.newaxis].astype(np.float32)
    interp.set_tensor(inp_det['index'], spec3)
    interp.invoke()
    out = interp.get_tensor(out_det['index'])
    tflite_preds.append(np.argmax(out))

# Agreement between full-precision and TFLite
agreement = np.mean(np.array(fp_preds) == np.array(tflite_preds))
print(f'FP vs TFLite agreement: {agreement*100:.1f}%  (target >98%)')

if agreement < 0.98:
    print('⚠️  Agreement below 98% — calibration dataset may be too small. Try 500+ samples.')
else:
    print('✅  Quantization quality OK')

## 5. Inspect model I/O shapes (must match app expectations)

In [ ]:
interp = tf.lite.Interpreter(model_path=TFLITE_OUT)
interp.allocate_tensors()

inp = interp.get_input_details()[0]
out = interp.get_output_details()[0]

print('INPUT')
print(f'  name  : {inp["name"]}')
print(f'  shape : {inp["shape"]}  — expected [1, 128, 128, 3]')
print(f'  dtype : {inp["dtype"]}')
print()
print('OUTPUT')
print(f'  name  : {out["name"]}')
print(f'  shape : {out["shape"]}  — expected [1, 4]')
print(f'  dtype : {out["dtype"]}')
print()
print('Class order:', {i: c for i, c in enumerate(CLASSES)})
print('(index 0 = snoring, 1 = breathing, 2 = silence, 3 = ambient)')

## 6. Download the model

In [ ]:
# Download to your local machine from Colab
from google.colab import files
files.download(TFLITE_OUT)
print('Downloading snore_classifier.tflite...')
print()
print('Next step:')
print('  Copy the downloaded file to:')
print('  mobile/assets/models/snore_classifier.tflite')
print('  (replace the 8-byte placeholder stub)')

## Done ✅
Place `snore_classifier.tflite` in `mobile/assets/models/` and rebuild the app.  
The `RecordScreen` Privacy Mode will automatically use it via `react-native-fast-tflite`.